In [1]:
"""
CNC OVERTIME MINIMIZATION - EXACT MILP (FULL DATASET)
Gurobi + Integer Job Splitting + Global Machine No-Overlap + HTML Gantt

REVISIONS (per supervisor feedback):
  1. BIG_M: 10**7 → 50,000
     Rationale: schedule horizon ~7*24*60 = 10,080 min; max Ca+Sb cannot exceed
     ~3x horizon ≈ 30,240 min. 50,000 is a safe tight bound, ~200x smaller than
     10**7. Tighter Big-M strengthens LP relaxation → better bounds → faster solve.

  2. Gurobi log file: model.Params.LogFile = 'gurobi_exact_full.log'
     Log is written to BASE_DIR for GitHub upload.

  3. TIME_LIMIT_SEC: kept at 21600 (6 hours).

  4. MIPFocus = 1 + NoRelHeurTime = 120
     Full model has ~20,000+ binary variables. Without a feasibility push,
     Gurobi can spend all 6 hours in LP relaxation without ever finding a
     feasible integer point (SolCount=0). MIPFocus=1 prioritises feasibility;
     NoRelHeurTime=120 runs 2 min of pure heuristic search at the very start.

  5. Robust SolCount=0 handling: code no longer crashes when no solution found.
     Previously `total_overtime.getValue()` raised an exception when SolCount=0.

  6. TARDINESS_LIMIT_DAYS stays at 1.5 (intentional project decision).
"""

import os
import math
import re
import time as time_module
import tracemalloc
from dataclasses import dataclass
from datetime import datetime, timedelta, time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np
import psutil
from gurobipy import Model, GRB, quicksum

import plotly.graph_objects as go

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 0. CONFIGURATION
# ============================================================

BASE_DIR = Path(r'C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact')


def find_input_file(patterns, description):
    for pattern in patterns:
        matches = sorted(BASE_DIR.glob(pattern))
        if matches:
            return matches[0]
    available = "\n".join([p.name for p in sorted(BASE_DIR.glob('*.xlsx'))])
    raise FileNotFoundError(
        f"Could not find {description}.\n"
        f"Searched patterns: {patterns}\n"
        f"Available Excel files:\n{available}"
    )


SHIPMENT_FILE      = find_input_file(['492-güncel sevkiyat*.xlsx', '492*güncel*sevkiyat*.xlsx', '*sevkiyat*.xlsx'], 'shipment file')
SDST_FILE          = find_input_file(['SDST*.xlsx', '*SDST*.xlsx'], 'SDST file')
MACHINE_GROUP_FILE = find_input_file(['machine_group_data*.xlsx', '*machine*group*.xlsx'], 'machine group file')

OUTPUT_SCHEDULE_FILE   = BASE_DIR / 'optimized_exact_gurobi_schedule_split.xlsx'
OUTPUT_VALIDATION_FILE = BASE_DIR / 'verification_validation_exact_gurobi_split.xlsx'
OUTPUT_GANTT_FILE      = BASE_DIR / 'exact_gurobi_gantt_split.html'
GUROBI_LOG_FILE        = BASE_DIR / 'gurobi_exact_full.log'   # REVISION 2

print('\nUsing files:')
print('SHIPMENT_FILE      =', SHIPMENT_FILE)
print('SDST_FILE          =', SDST_FILE)
print('MACHINE_GROUP_FILE =', MACHINE_GROUP_FILE)
print('GUROBI_LOG_FILE    =', GUROBI_LOG_FILE)

# Planning
PLANNING_START_HOUR = 7
DUE_TIME_HOUR       = 17
OVERTIME_START      = time(17, 0)
OVERTIME_END        = time(21, 0)
DISALLOW_START_IN_OVERTIME = True

# NOTE: stays at 1.5 — intentional project decision
TARDINESS_LIMIT_DAYS = 1.5
TARDINESS_LIMIT_MIN  = TARDINESS_LIMIT_DAYS * 24 * 60   # 2160 min

ALLOW_JOB_SPLITTING = True
MAX_SPLITS    = 2
MIN_SPLIT_QTY = 100

INITIAL_SETUP_MIN           = 10.0
SAME_DIAM_SETUP_MIN         = 5.0
DEFAULT_DIFF_DIAM_SETUP_MIN = 20.0

MIP_GAP        = 0.05
TIME_LIMIT_SEC = 21600   # 6 hours
OUTPUT_FLAG    = 1

# REVISION 1: tightened Big-M
# Old value: 10**7 (10,000,000)
# New value: 50,000  (~3x the 7-day schedule horizon of 10,080 min)
# Rationale: no start/finish/no-overlap term can exceed this bound.
# Tighter M → stronger LP relaxation → Gurobi finds bounds/solutions faster.
BIG_M = 50_000

EPS = 1e-5

# Global reference for validation helper (set in __main__)
GLOBAL_DATA = None


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def normalize_machine_name(x):
    if pd.isna(x): return None
    s = str(x).strip().replace(' ', '').replace(',', '.').replace('O', '0')
    s = s.replace('T3.', 'T.3.')
    if re.match(r'^T3\d+', s):
        s = s.replace('T3', 'T.3.', 1)
    return s


def excel_serial_to_datetime_date(x):
    if pd.isna(x): return None
    if isinstance(x, datetime): return x.date()
    if isinstance(x, (int, float, np.integer, np.floating)):
        return (datetime(1899, 12, 30) + timedelta(days=float(x))).date()
    parsed = pd.to_datetime(x, errors='coerce')
    return None if pd.isna(parsed) else parsed.date()


def excel_time_to_timedelta(x):
    if pd.isna(x): return None
    if isinstance(x, timedelta): return x
    if isinstance(x, datetime): return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, time): return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float, np.integer, np.floating)): return timedelta(days=float(x))
    parsed = pd.to_datetime(str(x), errors='coerce')
    return None if pd.isna(parsed) else timedelta(hours=parsed.hour, minutes=parsed.minute, seconds=parsed.second)


def combine_excel_date_time(date_value, time_value):
    d  = excel_serial_to_datetime_date(date_value)
    td = excel_time_to_timedelta(time_value)
    if d is None or td is None: return None
    return datetime.combine(d, time(0, 0)) + td


def minutes_between(start_dt, end_dt):
    if end_dt <= start_dt: end_dt += timedelta(days=1)
    return (end_dt - start_dt).total_seconds() / 60.0


def overlap_minutes(a_start, a_end, b_start, b_end):
    return max(0.0, (min(a_end, b_end) - max(a_start, b_start)).total_seconds() / 60.0)


def overtime_overlap_minutes(start_dt, end_dt):
    if end_dt <= start_dt: return 0.0
    total, day = 0.0, start_dt.date()
    while datetime.combine(day, time(0, 0)) <= end_dt:
        total += overlap_minutes(start_dt, end_dt,
                                 datetime.combine(day, OVERTIME_START),
                                 datetime.combine(day, OVERTIME_END))
        day += timedelta(days=1)
    return total


def parse_overtime_text(value):
    if pd.isna(value): return 0.0
    s = str(value).strip().lower()
    h = re.search(r'(\d+)\s*saat', s)
    m = re.search(r'(\d+)\s*dk', s)
    hours = int(h.group(1)) if h else 0
    mins  = int(m.group(1)) if m else 0
    if not h and not m:
        nums = re.findall(r'\d+', s)
        mins = int(nums[0]) if nums else 0
    return float(hours * 60 + mins)


def mode_or_first(series, default=None):
    s = series.dropna()
    if len(s) == 0: return default
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else s.iloc[0]


def dt_to_min(dt, planning_start): return (dt - planning_start).total_seconds() / 60.0
def min_to_dt(minutes, planning_start): return planning_start + timedelta(minutes=float(minutes))


# ============================================================
# 2. DATA CLASSES
# ============================================================

@dataclass
class Job:
    job_id: str
    part_no: str
    due_date: datetime
    quantity: int
    diameter: float
    eligible_groups_op10: list
    eligible_groups_op20: list


# ============================================================
# 3. LOAD INPUT DATA
# ============================================================

def load_shipment_operations(path):
    raw = pd.read_excel(path, sheet_name=0, header=5, usecols='D:N', engine='openpyxl')
    raw.columns = [str(c).strip().replace(':', '') for c in raw.columns]
    raw = raw.dropna(how='all')
    rename_map = {
        'Tarih': 'date', 'Başlangıç Saat': 'start_time', 'Bitiş Saat': 'finish_time',
        'Parça No': 'part_no', 'Makine No': 'machine',
        'CNC-1 operasyonu(piston)': 'op10_flag', 'CNC-2 operasyonu (saplama)': 'op20_flag',
        'Adet': 'quantity', 'Çap': 'diameter', 'Makine Grubu': 'machine_group',
        'Fazla mesai': 'overtime_text',
    }
    raw = raw.rename(columns=rename_map)
    raw = raw[raw['part_no'].notna()].copy()
    raw['machine']       = raw['machine'].apply(normalize_machine_name)
    raw['part_no']       = raw['part_no'].astype(int).astype(str)
    raw['quantity']      = pd.to_numeric(raw['quantity'], errors='coerce').fillna(0).astype(int)
    raw['diameter']      = pd.to_numeric(raw['diameter'], errors='coerce')
    raw['machine_group'] = pd.to_numeric(raw['machine_group'], errors='coerce').astype('Int64')
    raw['operation']     = np.where(raw.get('op10_flag').notna(), 10,
                           np.where(raw.get('op20_flag').notna(), 20, np.nan))
    raw = raw[raw['operation'].notna()].copy()
    raw['operation']     = raw['operation'].astype(int)
    raw['start_dt']      = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['start_time'])]
    raw['finish_dt']     = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['finish_time'])]
    raw['duration_min']  = [minutes_between(s, f) for s, f in zip(raw['start_dt'], raw['finish_dt'])]
    raw['unit_min']      = raw['duration_min'] / raw['quantity'].replace(0, np.nan)
    return raw


def load_orders_from_sayfa2(path, use_shipped_quantity=False):
    df = pd.read_excel(path, sheet_name=1, header=None, engine='openpyxl')
    jobs, current_date, in_block = [], None, False
    for _, row in df.iterrows():
        possible_date = None
        for val in row.tolist():
            if pd.isna(val): continue
            if isinstance(val, (int, float, np.integer, np.floating)) and 40000 < float(val) < 60000:
                possible_date = excel_serial_to_datetime_date(val); break
            if isinstance(val, datetime):
                possible_date = val.date(); break
        if possible_date is not None:
            current_date, in_block = possible_date, False; continue
        row_text = ' '.join([str(v).lower() for v in row.dropna().tolist()])
        if 'sipariş' in row_text and 'adet' in row_text:
            in_block = True; continue
        if in_block and current_date is not None:
            part      = row.iloc[4] if len(row) > 4 else None
            order_qty = row.iloc[5] if len(row) > 5 else None
            shipped   = row.iloc[6] if len(row) > 6 else None
            if pd.isna(part) or pd.isna(order_qty): continue
            try: part_no = str(int(part))
            except: continue
            qty_source = shipped if use_shipped_quantity and not pd.isna(shipped) else order_qty
            qty = int(round(float(qty_source)))
            if qty <= 0: continue
            jobs.append({'job_id': f'{current_date.isoformat()}__{part_no}',
                         'part_no': part_no,
                         'due_date': datetime.combine(current_date, time(DUE_TIME_HOUR, 0)),
                         'quantity': qty})
    orders = pd.DataFrame(jobs)
    if orders.empty: raise ValueError('Could not parse any order rows from Sayfa2.')
    before_n = len(orders)
    orders = orders.groupby(['job_id', 'part_no'], as_index=False).agg(
        quantity=('quantity', 'sum'), due_date=('due_date', 'min'))
    if len(orders) != before_n:
        print(f'Duplicate order rows aggregated: {before_n} -> {len(orders)} unique jobs')
    return orders


def load_machine_groups(path):
    mg = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    mg.columns = [str(c).strip() for c in mg.columns]
    mg = mg.dropna(subset=['Machine_number', 'Group']).copy()
    mg['Machine_number'] = mg['Machine_number'].apply(normalize_machine_name)
    mg['Group'] = pd.to_numeric(mg['Group'], errors='coerce').astype(int)
    g2m, m2g = defaultdict(list), {}
    for _, r in mg.iterrows():
        g2m[int(r['Group'])].append(r['Machine_number'])
        m2g[r['Machine_number']] = int(r['Group'])
    return dict(g2m), m2g


def load_sdst(path):
    sdst = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    sdst.columns = [str(c).strip() for c in sdst.columns]
    setup = {}
    for _, r in sdst.dropna(subset=['diam_from', 'to_diam', 'setup_time']).iterrows():
        setup[(float(r['diam_from']), float(r['to_diam']))] = float(r['setup_time'])
    return setup


def build_problem_data(shipment_file, sdst_file, machine_group_file, use_shipped_quantity=False):
    ops    = load_shipment_operations(shipment_file)
    orders = load_orders_from_sayfa2(shipment_file, use_shipped_quantity=use_shipped_quantity)
    g2m, m2g = load_machine_groups(machine_group_file)
    setup_dict = load_sdst(sdst_file)

    part_diam  = ops.groupby('part_no')['diameter'].agg(lambda x: mode_or_first(x, default=0)).to_dict()
    eligible   = defaultdict(lambda: defaultdict(set))
    for _, r in ops.iterrows():
        eligible[r['part_no']][int(r['operation'])].add(int(r['machine_group']))

    unit_time = {}
    for key, g in ops.dropna(subset=['unit_min', 'machine_group']).groupby(['part_no', 'operation', 'machine_group']):
        unit_time[(str(key[0]), int(key[1]), int(key[2]))] = float(g['unit_min'].median())
    fb_part_op  = ops.dropna(subset=['unit_min']).groupby(['part_no', 'operation'])['unit_min'].median().to_dict()
    fb_op       = ops.dropna(subset=['unit_min']).groupby(['operation'])['unit_min'].median().to_dict()
    global_unit = float(ops['unit_min'].dropna().median())

    all_groups = sorted(g2m.keys())
    jobs = []
    for _, r in orders.iterrows():
        pno  = str(r['part_no'])
        diam = float(part_diam.get(pno, ops['diameter'].dropna().median()))
        eg10 = sorted(list(eligible[pno].get(10, set(all_groups))))
        eg20 = sorted(list(eligible[pno].get(20, set(eg10 if eg10 else all_groups))))
        if not eg10: eg10 = all_groups
        if not eg20: eg20 = eg10
        jobs.append(Job(job_id=r['job_id'], part_no=pno, due_date=r['due_date'],
                        quantity=int(r['quantity']), diameter=diam,
                        eligible_groups_op10=eg10, eligible_groups_op20=eg20))

    ops['shipment_file_overtime_min'] = ops.get('overtime_text', pd.Series(dtype=object)).apply(parse_overtime_text)
    baseline_ot = float(ops['shipment_file_overtime_min'].sum())
    ps = datetime.combine(min([j.due_date.date() for j in jobs]), time(PLANNING_START_HOUR, 0))

    return {'ops': ops, 'baseline_overtime_min': baseline_ot, 'orders': orders, 'jobs': jobs,
            'group_to_machines': g2m, 'machine_to_group': m2g, 'setup_dict': setup_dict,
            'unit_time': unit_time, 'fallback_part_op': fb_part_op, 'fallback_op': fb_op,
            'global_unit': global_unit, 'planning_start': ps}


def get_unit_time(data, part_no, operation, group):
    k = (str(part_no), int(operation), int(group))
    if k in data['unit_time']: return data['unit_time'][k]
    k2 = (str(part_no), int(operation))
    if k2 in data['fallback_part_op']: return float(data['fallback_part_op'][k2])
    if int(operation) in data['fallback_op']: return float(data['fallback_op'][int(operation)])
    return data['global_unit']


def get_setup_time(data, from_diam, to_diam):
    if from_diam is None: return INITIAL_SETUP_MIN
    d1, d2 = float(from_diam), float(to_diam)
    if abs(d1 - d2) < 1e-9: return SAME_DIAM_SETUP_MIN
    return data['setup_dict'].get((d1, d2), DEFAULT_DIFF_DIAM_SETUP_MIN)


def common_eligible_groups(job):
    common = sorted(set(job.eligible_groups_op10) & set(job.eligible_groups_op20))
    return common if common else sorted(job.eligible_groups_op10)


# ============================================================
# 4b. WARM START LOADER
# ============================================================

WARM_START_FILE = BASE_DIR / 'heuristic_warm_start.json'


def _load_warm_start(active, qty, z_group, qty_group, x, S, C, proc_time,
                     job_finish, makespan,
                     arc, first, last, ot_vars, start_side,
                     allowed_groups, elig_mach, job_by_id, arc_keys,
                     ot_windows, data):
    """
    Warm start: SADECE binary/integer karar değişkenlerine .Start verir:
      active, qty, z_group, qty_group, x, arc, first, last

    Continuous / türetilmiş değişkenlere (S, C, proc_time, ot_vars,
    start_side, job_finish, makespan) KESİNLİKLE dokunulmaz. Bunlar modelde
    sıkı eşitliklerle bağlı (örn. C == S + proc_time, proc_time == unit*qty,
    job_finish >= C). Dışarıdan verilen yaklaşık float değer bu eşitlikleri
    çok küçük bir farkla bile ihlal edebiliyordu (örn. R1002'de 0.00003 dk
    sapma) ve Gurobi bu yüzden partial solution'ı incumbent yapmıyordu.

    Bu değişkenleri boş bırakınca Gurobi, sabitlenen active/qty/z_group/
    qty_group/x/arc/first/last üzerinden geri kalan continuous değerleri
    kendi LP çözücüsüyle TUTARLI ve EŞİTLİKLERE TAM UYAN şekilde hesaplar.

    ÖNEMLİ: group ataması, warm start JSON'undaki 'group' alanından DEĞİL,
    görevin atandığı GERÇEK makinenin (machine) ait olduğu gruptan
    (data['machine_to_group']) türetilir. Bu sayede x (makine ataması) ile
    z_group (grup ataması) birbiriyle tutarlı olur.
    """
    import json

    if not WARM_START_FILE.exists():
        print(f'\nWARM START: {WARM_START_FILE} bulunamadı — warm start atlanıyor.')
        print('  → Önce heuristic_final.py çalıştırın.')
        return None

    with open(WARM_START_FILE, 'r', encoding='utf-8') as f:
        warm = json.load(f)

    heuristic_ot = sum(ti.get('overtime_min', 0.0) for ti in warm['tasks'])
    set_count    = 0
    machine_to_group = data['machine_to_group']

    # ── 1. active ────────────────────────────────────────────
    for jid, qs in warm['splits'].items():
        if jid not in job_by_id:
            continue
        n_splits = len(qs)
        for r in [1, 2]:
            if (jid, r) in active:
                active[jid, r].Start = 1 if r <= n_splits else 0
                set_count += 1

    # ── 2. split -> gerçek group (atanan makineden türetilir) ──
    # Bir split'in Op10 ve Op20'si farklı makinede olabilir ama exact modelde
    # ikisi de aynı z_group[jid,sid,g] tarafından kısıtlanıyor. Bu yüzden
    # tek bir group seçilmeli — Op10'un makinesinin grubunu esas alıyoruz.
    split_group = {}
    split_qty   = {}
    for ti in warm['tasks']:
        jid, sid, op, mach = ti['job_id'], ti['split_id'], ti['operation'], ti['machine']
        key = (jid, sid)
        split_qty[key] = ti.get('qty', warm['splits'].get(jid, [0])[sid - 1] if sid - 1 < len(warm['splits'].get(jid, [])) else 0)
        if op == 10 and mach in machine_to_group:
            split_group[key] = machine_to_group[mach]
        elif key not in split_group and mach in machine_to_group:
            split_group[key] = machine_to_group[mach]

    # ── 3. qty, z_group, qty_group — split_group ile tutarlı ──
    for (jid, sid), g_sel in split_group.items():
        if jid not in allowed_groups:
            continue
        q_val = int(round(split_qty.get((jid, sid), 0)))

        if (jid, sid) in qty:
            qty[jid, sid].Start = q_val
            set_count += 1

        for g in allowed_groups[jid]:
            if (jid, sid, g) in z_group:
                z_group[jid, sid, g].Start = 1 if g == g_sel else 0
                set_count += 1
            if (jid, sid, g) in qty_group:
                qty_group[jid, sid, g].Start = q_val if g == g_sel else 0
                set_count += 1

    # ── 4. x (makine ataması) + first/last default 0 ─────────
    # task_machine_map: t -> (mach, start_min) — SADECE sıralama amaçlı,
    # Gurobi değişkenine değil, kendi sıralama mantığımıza yardımcı veri.
    task_machine_map = {}
    for ti in warm['tasks']:
        jid  = ti['job_id']
        sid  = ti['split_id']
        op   = ti['operation']
        mach = ti['machine']
        t    = (jid, sid, op)
        task_machine_map[t] = (mach, ti['start_min'])

        for m in elig_mach.get(t, []):
            if (jid, sid, op, m) in x:
                x[jid, sid, op, m].Start = 1 if m == mach else 0
                set_count += 1
            # task bu makineye atanmadıysa first/last kesinlikle 0
            if m != mach:
                tid_other = (jid, sid, op, m)
                if tid_other in first:
                    first[tid_other].Start = 0
                    set_count += 1
                if tid_other in last:
                    last[tid_other].Start = 0
                    set_count += 1

    # ── 5. arc, first, last — makine üzerindeki gerçek sıra ────
    # Heuristic'in start_min sırasına göre ardışık görevler arc=1,
    # ilk görev first=1, son görev last=1 olarak işaretlenir.
    # Bu sadece SIRALAMA bilgisi taşır (continuous zaman değeri değil),
    # bu yüzden floating-point tutarlılık riski taşımaz.
    machine_tasks = {}
    for t, (mach, s_m) in task_machine_map.items():
        machine_tasks.setdefault(mach, []).append((s_m, t))
    for mach in machine_tasks:
        machine_tasks[mach].sort(key=lambda pair: pair[0])

    arc_keys_set = set(arc_keys)

    # Önce TÜM arc_keys'i 0'a set et (default) — hiçbir arc boş kalmasın
    for key in arc_keys:
        arc[key].Start = 0
        set_count += 1

    # Sonra her makinedeki ardışık görev çiftlerini 1 yap
    for mach, sorted_tasks in machine_tasks.items():
        tasks_on_m = [t for _, t in sorted_tasks]
        for idx, t in enumerate(tasks_on_m):
            jid, sid, op = t
            tid = (jid, sid, op, mach)
            if tid in first:
                first[tid].Start = 1 if idx == 0 else 0
                set_count += 1
            if tid in last:
                last[tid].Start = 1 if idx == len(tasks_on_m) - 1 else 0
                set_count += 1
        for idx in range(len(tasks_on_m) - 1):
            t_i = tasks_on_m[idx]
            t_j = tasks_on_m[idx + 1]
            key = (t_i[0], t_i[1], t_i[2], t_j[0], t_j[1], t_j[2], mach)
            if key in arc_keys_set:
                arc[key].Start = 1

    # ── S, C, proc_time, ot_vars, start_side, job_finish, makespan ──
    # BİLİNÇLİ OLARAK DOKUNULMUYOR. Gurobi bunları active/qty/z_group/
    # qty_group/x/arc/first/last sabitlerinden kendi LP'siyle, eşitliklere
    # tam uyacak şekilde türetir.

    print(f'\nWARM START yüklendi (SADECE binary/integer: active, qty, z_group, qty_group, x, arc, first, last):')
    print(f'  Kaynak        : {WARM_START_FILE}')
    print(f'  Initial OT    : {heuristic_ot:.1f} dk  (referans değer, Incumbent garantisi değil)')
    print(f'  .Start atanan : {set_count} değişken')
    print(f'  group ataması, atanan makinenin gerçek grubundan türetildi (x ile tutarlı)')
    print(f'  arc/first/last, makine üzerindeki gerçek start_min sırasından türetildi (x ile tutarlı)')
    print(f'  S, C, proc_time, ot_vars, job_finish, makespan: Gurobi kendi hesaplayacak (dokunulmadı).')

    return heuristic_ot


# ============================================================
# 5. EXACT GUROBI MODEL
# ============================================================

def build_overtime_windows(planning_start, latest_due):
    end = latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1)
    windows, day = [], planning_start.date()
    while datetime.combine(day, time(0, 0)) <= end:
        l = dt_to_min(datetime.combine(day, OVERTIME_START), planning_start)
        u = dt_to_min(datetime.combine(day, OVERTIME_END),   planning_start)
        if u > 0: windows.append((max(0.0, l), u, day.isoformat()))
        day += timedelta(days=1)
    return windows


def solve_exact_gurobi(data):
    jobs = data['jobs']

    job_ids = [j.job_id for j in jobs]
    if len(job_ids) != len(set(job_ids)):
        vc = pd.Series(job_ids).value_counts()
        raise ValueError(f'Duplicate job_ids: {vc[vc>1].index.tolist()[:20]}')

    machines        = sorted(data['machine_to_group'].keys())
    planning_start  = data['planning_start']
    latest_due      = max(j.due_date for j in jobs)
    ot_windows      = build_overtime_windows(planning_start, latest_due)
    split_ids       = list(range(1, (MAX_SPLITS if ALLOW_JOB_SPLITTING else 1) + 1))
    tasks           = [(j.job_id, r, op) for j in jobs for r in split_ids for op in [10, 20]]
    job_by_id       = {j.job_id: j for j in jobs}
    allowed_groups  = {j.job_id: common_eligible_groups(j) for j in jobs}

    elig_mach = {}
    for j in jobs:
        for r in split_ids:
            for op in [10, 20]:
                ms = []
                for g in allowed_groups[j.job_id]:
                    ms.extend(data['group_to_machines'].get(g, []))
                elig_mach[(j.job_id, r, op)] = sorted(set(ms))

    horizon = dt_to_min(latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1), planning_start)

    print(f'\nSchedule horizon : {horizon:.0f} min  ({horizon/60:.1f} h)')
    print(f'Big-M used       : {BIG_M}  (revised from 10**7 = {10**7}; {10**7//BIG_M}x tighter)')
    print(f'Tasks            : {len(tasks)}')

    M = BIG_M  # use the tightened value

    model = Model('Exact_Full')

    # REVISION 2: Gurobi log file
    model.Params.LogFile    = str(GUROBI_LOG_FILE)

    model.Params.MIPGap     = MIP_GAP
    model.Params.OutputFlag = OUTPUT_FLAG
    if TIME_LIMIT_SEC:
        model.Params.TimeLimit = TIME_LIMIT_SEC

    # REVISION 4: feasibility-first settings
    # Without these, full model often returns 0 solutions in 6h.
    # MIPFocus=1 tells Gurobi to prioritise finding ANY feasible solution.
    # NoRelHeurTime=120 runs 2 min of pure integer heuristics before LP relaxation.
    model.Params.MIPFocus      = 1    # 0=balanced, 1=feasibility, 2=optimality, 3=bound
    model.Params.NoRelHeurTime = 120  # seconds of no-relaxation heuristic at start

    # Variables
    active    = model.addVars([(j.job_id, r) for j in jobs for r in split_ids], vtype=GRB.BINARY,  name='active')
    qty       = model.addVars([(j.job_id, r) for j in jobs for r in split_ids], vtype=GRB.INTEGER, lb=0, name='qty')
    z_group   = model.addVars([(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_groups[j.job_id]], vtype=GRB.BINARY,  name='z_group')
    qty_group = model.addVars([(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_groups[j.job_id]], vtype=GRB.INTEGER, lb=0, name='qty_group')
    x         = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='x')
    S         = model.addVars(tasks, lb=0.0, name='S')
    C         = model.addVars(tasks, lb=0.0, name='C')
    proc_time = model.addVars(tasks, lb=0.0, name='proc_time')

    arc_keys = [(t[0], t[1], t[2], u[0], u[1], u[2], m)
                for t in tasks for u in tasks if t != u
                for m in set(elig_mach[t]) & set(elig_mach[u])]
    arc   = model.addVars(arc_keys, vtype=GRB.BINARY, name='arc')
    first = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='first')
    last  = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='last')

    job_finish = model.addVars([j.job_id for j in jobs], lb=0.0, name='job_finish')
    makespan   = model.addVar(lb=0.0, name='makespan')

    ot_vars    = {(t, wi): model.addVar(lb=0.0, name=f'ot_{t[0]}_{t[1]}_{t[2]}_{wi}')
                  for t in tasks for wi in range(len(ot_windows))}
    start_side = {}
    if DISALLOW_START_IN_OVERTIME:
        start_side = {(t, wi): model.addVar(vtype=GRB.BINARY, name=f'ss_{t[0]}_{t[1]}_{t[2]}_{wi}')
                      for t in tasks for wi in range(len(ot_windows))}

    # Split quantity constraints
    for j in jobs:
        jid, Q = j.job_id, int(j.quantity)
        min_q  = min(MIN_SPLIT_QTY, Q)
        model.addConstr(active[jid, 1] == 1)
        model.addConstr(quicksum(qty[jid, r] for r in split_ids) == Q)
        for r in split_ids:
            if r > 1:
                model.addConstr(active[jid, r] <= active[jid, r-1])
                if Q < 2 * MIN_SPLIT_QTY:
                    model.addConstr(active[jid, r] == 0)
            model.addConstr(qty[jid, r] <= Q * active[jid, r])
            model.addConstr(qty[jid, r] >= min_q * active[jid, r])
            model.addConstr(quicksum(z_group[jid, r, g] for g in allowed_groups[jid]) == active[jid, r])
            model.addConstr(quicksum(qty_group[jid, r, g] for g in allowed_groups[jid]) == qty[jid, r])
            for g in allowed_groups[jid]:
                model.addConstr(qty_group[jid, r, g] <= Q * z_group[jid, r, g])
                model.addConstr(qty_group[jid, r, g] <= qty[jid, r])
                model.addConstr(qty_group[jid, r, g] >= qty[jid, r] - Q * (1 - z_group[jid, r, g]))

    # Assignment constraints
    for t in tasks:
        jid, r, op = t
        model.addConstr(quicksum(x[jid, r, op, m] for m in elig_mach[t]) == active[jid, r])
        for m in elig_mach[t]:
            gm = data['machine_to_group'][m]
            if gm in allowed_groups[jid]:
                model.addConstr(x[jid, r, op, m] <= z_group[jid, r, gm])
            else:
                model.addConstr(x[jid, r, op, m] == 0)

    # Processing time, completion, precedence
    for t in tasks:
        jid, r, op = t
        job = job_by_id[jid]
        p_expr = quicksum(get_unit_time(data, job.part_no, op, g) * qty_group[jid, r, g]
                          for g in allowed_groups[jid])
        model.addConstr(proc_time[t] == p_expr)
        model.addConstr(C[t] == S[t] + proc_time[t])
        model.addConstr(S[t] <= M * active[jid, r])
        model.addConstr(C[t] <= M * active[jid, r])

    for j in jobs:
        for r in split_ids:
            model.addConstr(S[j.job_id, r, 20] >= C[j.job_id, r, 10] - M * (1 - active[j.job_id, r]))

    for j in jobs:
        due_m = dt_to_min(j.due_date, planning_start)
        for r in split_ids:
            model.addConstr(job_finish[j.job_id] >= C[j.job_id, r, 20])
        model.addConstr(job_finish[j.job_id] <= due_m + TARDINESS_LIMIT_MIN)
        model.addConstr(makespan >= job_finish[j.job_id])

    # Machine sequencing
    for m in machines:
        pm = [t for t in tasks if m in elig_mach[t]]
        if not pm: continue
        model.addConstr(quicksum(first[t[0], t[1], t[2], m] for t in pm) <= 1)
        model.addConstr(quicksum(last[t[0], t[1], t[2], m]  for t in pm) <= 1)
        for t in pm:
            tid = (t[0], t[1], t[2], m)
            pred_a = [arc[u[0], u[1], u[2], t[0], t[1], t[2], m] for u in pm if u != t and (u[0], u[1], u[2], t[0], t[1], t[2], m) in arc]
            succ_a = [arc[t[0], t[1], t[2], u[0], u[1], u[2], m] for u in pm if u != t and (t[0], t[1], t[2], u[0], u[1], u[2], m) in arc]
            model.addConstr(quicksum(pred_a) + first[tid] == x[tid])
            model.addConstr(quicksum(succ_a) + last[tid]  == x[tid])
            model.addConstr(first[tid] <= x[tid])
            model.addConstr(last[tid]  <= x[tid])
            model.addConstr(S[t] >= INITIAL_SETUP_MIN - M * (1 - first[tid]))

    for key in arc_keys:
        i_jid, i_r, i_op, j_jid, j_r, j_op, m = key
        model.addConstr(arc[key] <= x[i_jid, i_r, i_op, m])
        model.addConstr(arc[key] <= x[j_jid, j_r, j_op, m])
        s_ij = get_setup_time(data, job_by_id[i_jid].diameter, job_by_id[j_jid].diameter)
        model.addConstr(S[(j_jid, j_r, j_op)] >= C[(i_jid, i_r, i_op)] + s_ij - M * (1 - arc[key]))

    # Overtime overlap
    for t in tasks:
        jid, r, op = t
        for wi, (l, u, _) in enumerate(ot_windows):
            ms_v = model.addVar(lb=-M, ub=M, name=f'ms_{jid}_{r}_{op}_{wi}')
            mf_v = model.addVar(lb=-M, ub=M, name=f'mf_{jid}_{r}_{op}_{wi}')
            df_v = model.addVar(lb=-M, ub=M, name=f'df_{jid}_{r}_{op}_{wi}')
            model.addGenConstrMax(ms_v, [S[t]], constant=float(l))
            model.addGenConstrMin(mf_v, [C[t]], constant=float(u))
            model.addConstr(df_v == mf_v - ms_v)
            model.addGenConstrMax(ot_vars[(t, wi)], [df_v], constant=0.0)
            model.addConstr(ot_vars[(t, wi)] <= (u - l) * active[jid, r])
            if DISALLOW_START_IN_OVERTIME:
                b = start_side[(t, wi)]
                model.addConstr(S[t] <= float(l) + M * b + M * (1 - active[jid, r]))
                model.addConstr(S[t] >= float(u) - M * (1 - b) - M * (1 - active[jid, r]))

    total_overtime = quicksum(ot_vars[(t, wi)] for t in tasks for wi in range(len(ot_windows)))

    # Objective: minimize total overtime only — same as heuristic objective
    model.setObjective(total_overtime, GRB.MINIMIZE)
    model.update()

    # Count variable types
    n_binary = sum(1 for v in model.getVars() if v.VType == GRB.BINARY)
    n_int    = sum(1 for v in model.getVars() if v.VType == GRB.INTEGER)
    n_cont   = sum(1 for v in model.getVars() if v.VType == GRB.CONTINUOUS)

    print(f'\nModel size:')
    print(f'  Total variables : {model.NumVars:,}')
    print(f'  Binary          : {n_binary:,}')
    print(f'  Integer         : {n_int:,}')
    print(f'  Continuous      : {n_cont:,}')
    print(f'  Constraints     : {model.NumConstrs:,}')
    print(f'  GenConstrs      : {model.NumGenConstrs:,}')
    print(f'  Arc keys        : {len(arc_keys):,}')
    print(f'\nGurobi log       : {GUROBI_LOG_FILE}')
    print(f'Time limit       : {TIME_LIMIT_SEC} sec ({TIME_LIMIT_SEC/3600:.0f} h)')
    print(f'MIPFocus         : 1 (feasibility-first)')
    print(f'NoRelHeurTime    : 120 sec')

    # --- WARM START (initial solution) — best effort, never crashes ---
    # Eğer warm start herhangi bir nedenle constraint ihlali yaratırsa veya
    # hata verirse, sessizce atlanır ve Gurobi MIPFocus=1 + NoRelHeurTime=120
    # ile kendi başına feasible solution arar. Sonuç HER DURUMDA üretilir.
    heuristic_ot = None
    try:
        heuristic_ot = _load_warm_start(
            active=active, qty=qty, z_group=z_group, qty_group=qty_group,
            x=x, S=S, C=C, proc_time=proc_time,
            job_finish=job_finish, makespan=makespan,
            arc=arc, first=first, last=last, ot_vars=ot_vars, start_side=start_side,
            allowed_groups=allowed_groups, elig_mach=elig_mach,
            job_by_id=job_by_id, arc_keys=arc_keys, ot_windows=ot_windows,
            data=data,
        )
    except Exception as e:
        print(f'\nWARM START HATASI (atlanıyor, Gurobi sıfırdan arayacak): {e}')
        heuristic_ot = None
    # --------------------------------------------------------------------

    model.optimize()

    # Eğer Gurobi warm start'ı reddettiyse (MIP start rejected / no new incumbent),
    # bu sadece bir UYARI'dır — model.optimize() yine de normal şekilde devam eder
    # ve kendi NoRel heuristic'i + branch&bound ile feasible solution arar.
    # Bu yüzden SolCount=0 olması garanti DEĞİLDİR; sadece warm start kullanılmamış olur.

    # REVISION 5: robust status handling — no crash on SolCount=0
    status = model.Status
    sol_count = model.SolCount

    print(f'\nGurobi Status  : {status}')
    print(f'Solution Count : {sol_count}')
    print(f'Warm start OT  : {heuristic_ot:.1f} dk (referans)' if heuristic_ot is not None else 'Warm start    : kullanılmadı')

    if sol_count > 0:
        gap_pct = model.MIPGap * 100
        print(f'Objective (Gurobi Incumbent) : {model.ObjVal:.4f} dk')
        print(f'Best Bound                   : {model.ObjBound:.4f} dk')
        print(f'MIP Gap                      : {gap_pct:.2f}%')
        if heuristic_ot is not None:
            print(f'Karşılaştırma: heuristic initial OT = {heuristic_ot:.1f} dk vs Gurobi incumbent = {model.ObjVal:.1f} dk')
        ot_val     = total_overtime.getValue()
        mk_val     = makespan.X
    else:
        print('WARNING: No feasible solution found within time limit.')
        print('Bu durumda da heuristic initial solution değeri raporlanabilir:')
        if heuristic_ot is not None:
            print(f'  Heuristic initial OT = {heuristic_ot:.1f} dk (Gurobi bunu incumbent yapamadı)')
        ot_val, mk_val = None, None

    if status == GRB.INFEASIBLE:
        print('Model is INFEASIBLE. Computing IIS...')
        model.computeIIS()
        iis_path = BASE_DIR / 'model_iis.ilp'
        model.write(str(iis_path))
        print('IIS written to:', iis_path)

    return {
        'model': model,
        'active': active, 'qty': qty, 'z_group': z_group, 'qty_group': qty_group,
        'x': x, 'S': S, 'C': C, 'proc_time': proc_time,
        'arc': arc, 'first': first, 'last': last,
        'job_finish': job_finish, 'makespan': makespan,
        'ot_vars': ot_vars, 'overtime_windows': ot_windows,
        'tasks': tasks, 'machines': machines, 'split_ids': split_ids,
        'allowed_groups': allowed_groups, 'elig_mach': elig_mach, 'job_by_id': job_by_id,
        'arc_keys': arc_keys,
        'objective_total_overtime': ot_val,
        'objective_makespan': mk_val,
        'sol_count': sol_count,
        'heuristic_initial_ot': heuristic_ot,
        'model_stats': {'n_vars': model.NumVars, 'n_binary': n_binary,
                        'n_int': n_int, 'n_cont': n_cont,
                        'n_constrs': model.NumConstrs, 'n_arc_keys': len(arc_keys)},
    }


# ============================================================
# 6. SOLUTION EXTRACTION  (unchanged from original)
# ============================================================

def value(v):
    try: return float(v.X)
    except: return float(v)


def extract_solution_tables(data, sol):
    if sol['sol_count'] == 0:
        print('No solution to extract.')
        return pd.DataFrame(), pd.DataFrame()

    job_by_id = sol['job_by_id']
    split_ids = sol['split_ids']
    tasks     = sol['tasks']
    active, qty, z_group = sol['active'], sol['qty'], sol['z_group']
    x, S, C, proc_time  = sol['x'], sol['S'], sol['C'], sol['proc_time']
    arc, first, last     = sol['arc'], sol['first'], sol['last']
    planning_start = data['planning_start']

    split_rows, selected_qty, selected_group = [], {}, {}
    for j in data['jobs']:
        for r in split_ids:
            if value(active[j.job_id, r]) > 0.5:
                q = round(value(qty[j.job_id, r]))
                g_sel = next((g for g in sol['allowed_groups'][j.job_id] if value(z_group[j.job_id, r, g]) > 0.5), None)
                selected_qty[(j.job_id, r)]   = int(q)
                selected_group[(j.job_id, r)] = g_sel
                split_rows.append({'job_id': j.job_id, 'part_no': j.part_no, 'split_id': r,
                                   'quantity': int(q), 'selected_group': g_sel,
                                   'original_quantity': j.quantity, 'due_date': j.due_date})

    selected_machine = {}
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5: continue
        for m in sol['elig_mach'][t]:
            if (jid, r, op, m) in x and value(x[jid, r, op, m]) > 0.5:
                selected_machine[t] = m; break

    pred_task, setup_by_task, first_flag = {}, {}, {}
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5: continue
        m = selected_machine.get(t)
        if m is None: continue
        first_flag[t] = value(first[jid, r, op, m]) > 0.5
        if first_flag[t]:
            pred_task[t], setup_by_task[t] = None, INITIAL_SETUP_MIN
        else:
            pred, setup = None, 0.0
            for key, var in arc.items():
                i_jid, i_r, i_op, j_jid, j_r, j_op, mm = key
                if mm == m and (j_jid, j_r, j_op) == t and value(var) > 0.5:
                    pred  = (i_jid, i_r, i_op)
                    setup = get_setup_time(data, job_by_id[i_jid].diameter, job_by_id[j_jid].diameter)
                    break
            pred_task[t], setup_by_task[t] = pred, setup

    rows = []
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5: continue
        job  = job_by_id[jid]
        m    = selected_machine.get(t)
        if m is None: continue
        s_min  = value(S[t]); f_min = value(C[t])
        p_min  = value(proc_time[t])
        setup  = setup_by_task.get(t, 0.0)
        ss_min = max(0.0, s_min - setup)
        ot_min = overtime_overlap_minutes(min_to_dt(s_min, planning_start), min_to_dt(f_min, planning_start))
        pred   = pred_task.get(t)
        rows.append({
            'job_id': jid, 'part_no': job.part_no, 'split_id': r,
            'split_count': sum(1 for rr in split_ids if value(active[jid, rr]) > 0.5),
            'quantity': selected_qty.get((jid, r), 0), 'diameter': job.diameter,
            'group': selected_group.get((jid, r)), 'operation': op, 'machine': m,
            'start_min': s_min, 'finish_min': f_min, 'setup_start_min': ss_min,
            'setup_finish_min': s_min, 'processing_min': p_min, 'setup_min': setup,
            'overtime_min': ot_min,
            'start': min_to_dt(s_min, planning_start), 'finish': min_to_dt(f_min, planning_start),
            'setup_start': min_to_dt(ss_min, planning_start), 'setup_finish': min_to_dt(s_min, planning_start),
            'prev_task': 'SRC' if pred is None else f"{job_by_id[pred[0]].part_no} S{pred[1]} Op{pred[2]}",
            'is_first_on_machine': pred is None,
            'due_date': job.due_date,
        })

    schedule_df = pd.DataFrame(rows)
    if not schedule_df.empty:
        schedule_df = schedule_df.sort_values(['machine', 'start_min']).reset_index(drop=True)
        schedule_df['global_sequence_on_machine'] = schedule_df.groupby('machine').cumcount() + 1
    return schedule_df, pd.DataFrame(split_rows)


def check_machine_global_nooverlap(schedule_df, tolerance_min=1e-5):
    rows = []
    for machine, g in schedule_df.sort_values(['machine', 'start_min']).groupby('machine'):
        recs = g.to_dict('records')
        for i in range(len(recs) - 1):
            cur, nxt = recs[i], recs[i+1]
            req = cur['finish_min'] + get_setup_time(GLOBAL_DATA, cur['diameter'], nxt['diameter'])
            rows.append({
                'machine': machine,
                'current_task': f"{cur['part_no']} S{cur['split_id']} Op{cur['operation']}",
                'next_task': f"{nxt['part_no']} S{nxt['split_id']} Op{nxt['operation']}",
                'current_finish_min': cur['finish_min'],
                'next_start_min': nxt['start_min'],
                'setup_between_min': get_setup_time(GLOBAL_DATA, cur['diameter'], nxt['diameter']),
                'slack_min': nxt['start_min'] - req,
                'processing_overlap_min': max(0.0, cur['finish_min'] - nxt['start_min']),
                'Feasible': (nxt['start_min'] - req) >= -tolerance_min,
            })
    return pd.DataFrame(rows) if rows else pd.DataFrame([{'status': 'NO PAIRS', 'Feasible': True}])


def build_validation_tables(data, sol, schedule_df, split_df):
    if schedule_df.empty:
        return {'Feasibility_Summary': pd.DataFrame([{'Check': 'No solution', 'All_Feasible': False, 'Violations': 0}])}

    jobs, split_ids = data['jobs'], sol['split_ids']
    active, qty, S, C, proc_time_v, x = sol['active'], sol['qty'], sol['S'], sol['C'], sol['proc_time'], sol['x']
    job_by_id = sol['job_by_id']

    q_rows, a_rows, c_rows, p_rows, t_rows = [], [], [], [], []
    nooverlap_df = check_machine_global_nooverlap(schedule_df)

    for j in jobs:
        total_q = sum(round(value(qty[j.job_id, r])) for r in split_ids)
        q_rows.append({'job_id': j.job_id, 'part_no': j.part_no,
                       'original_quantity': j.quantity, 'sum_split_quantity': total_q,
                       'Feasible': abs(total_q - j.quantity) <= EPS})
        finish_vals = [value(C[j.job_id, r, 20]) for r in split_ids if value(active[j.job_id, r]) > 0.5]
        jf   = max(finish_vals) if finish_vals else 0.0
        due  = dt_to_min(j.due_date, data['planning_start'])
        viol = max(0.0, jf - (due + TARDINESS_LIMIT_MIN))
        t_rows.append({'job_id': j.job_id, 'part_no': j.part_no, 'job_finish_min': jf,
                       'due_min': due, 'tardiness_limit_min': TARDINESS_LIMIT_MIN,
                       'tardiness_limit_violation_min': viol, 'Feasible': viol <= EPS})

    for t in sol['tasks']:
        jid, r, op = t
        a  = value(active[jid, r])
        as_ = sum(value(x[jid, r, op, m]) for m in sol['elig_mach'][t] if (jid, r, op, m) in x)
        a_rows.append({'job_id': jid, 'split_id': r, 'operation': op, 'active': a,
                       'assignment_sum': as_, 'Feasible': abs(as_ - a) <= EPS})
        lhs = value(C[t]); rhs = value(S[t]) + value(proc_time_v[t])
        c_rows.append({'job_id': jid, 'split_id': r, 'operation': op, 'C': lhs,
                       'S_plus_proc': rhs, 'Deviation': abs(lhs - rhs), 'Feasible': abs(lhs - rhs) <= EPS})

    for j in data['jobs']:
        for r in split_ids:
            if value(active[j.job_id, r]) > 0.5:
                slack = value(S[j.job_id, r, 20]) - value(C[j.job_id, r, 10])
                p_rows.append({'job_id': j.job_id, 'split_id': r,
                               'start_op20': value(S[j.job_id, r, 20]),
                               'finish_op10': value(C[j.job_id, r, 10]),
                               'slack_min': slack, 'Feasible': slack >= -EPS})

    dfs = {'Split_Quantity_Check': pd.DataFrame(q_rows),
           'Assignment_Check': pd.DataFrame(a_rows),
           'Completion_Check': pd.DataFrame(c_rows),
           'Precedence_Check': pd.DataFrame(p_rows),
           'Tardiness_Check':  pd.DataFrame(t_rows),
           'Machine_Global_NoOverlap': nooverlap_df}
    summary = [{'Check': n, 'All_Feasible': bool(df['Feasible'].all()),
                 'Violations': int((~df['Feasible']).sum())}
               for n, df in dfs.items() if 'Feasible' in df.columns]
    dfs['Feasibility_Summary'] = pd.DataFrame(summary)
    return dfs


def build_machine_summary(schedule_df):
    return schedule_df.groupby('machine').agg(
        total_processing_min=('processing_min', 'sum'),
        total_setup_min=('setup_min', 'sum'),
        total_overtime_min=('overtime_min', 'sum'),
        first_start=('start', 'min'), last_finish=('finish', 'max'),
        task_count=('job_id', 'count'),
    ).reset_index().sort_values('total_overtime_min', ascending=False)


def build_job_summary(schedule_df, data):
    jmap = {j.job_id: j for j in data['jobs']}
    comp = schedule_df[schedule_df['operation'] == 20].groupby('job_id')['finish'].max().reset_index()
    rows = []
    for _, r in comp.iterrows():
        job = jmap[r['job_id']]; finish = r['finish']
        tard = max(0.0, (finish - job.due_date).total_seconds() / 60.0)
        rows.append({'job_id': job.job_id, 'part_no': job.part_no, 'quantity': job.quantity,
                     'due_date': job.due_date, 'completion': finish, 'tardiness_min': tard,
                     'tardiness_limit_violation_min': max(0.0, (finish - (job.due_date + timedelta(minutes=TARDINESS_LIMIT_MIN))).total_seconds() / 60.0)})
    return pd.DataFrame(rows).sort_values(['due_date', 'part_no'])


def export_excel_outputs(data, sol, schedule_df, split_df, validation_dfs, cpu_sec, peak_mb):
    model = sol['model']
    stats = sol['model_stats']

    metrics = pd.DataFrame([{
        'gurobi_status':        model.Status,
        'sol_count':            sol['sol_count'],
        'objective_value':      round(model.ObjVal, 4) if sol['sol_count'] > 0 else None,
        'best_bound':           round(model.ObjBound, 4) if sol['sol_count'] > 0 else None,
        'mip_gap_pct':          round(model.MIPGap * 100, 2) if sol['sol_count'] > 0 else None,
        'total_overtime_min':   round(schedule_df['overtime_min'].sum(), 2) if not schedule_df.empty else None,
        'baseline_overtime_min': data.get('baseline_overtime_min', 0.0),
        'heuristic_initial_ot_min': round(sol.get('heuristic_initial_ot'), 2) if sol.get('heuristic_initial_ot') is not None else None,
        'warm_start_used':      sol.get('heuristic_initial_ot') is not None,
        'cpu_time_sec':         round(cpu_sec, 2),
        'cpu_time_hours':       round(cpu_sec / 3600, 4),
        'peak_memory_mb':       round(peak_mb, 2),
        'peak_memory_gb':       round(peak_mb / 1024, 3),
        'big_m_used':           BIG_M,
        'tardiness_limit_days': TARDINESS_LIMIT_DAYS,
        'time_limit_sec':       TIME_LIMIT_SEC,
        'n_vars_total':         stats['n_vars'],
        'n_vars_binary':        stats['n_binary'],
        'n_vars_integer':       stats['n_int'],
        'n_vars_continuous':    stats['n_cont'],
        'n_constraints':        stats['n_constrs'],
        'n_arc_keys':           stats['n_arc_keys'],
        'gurobi_log':           str(GUROBI_LOG_FILE),
    }])

    with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
        if not schedule_df.empty:
            schedule_df.to_excel(writer, sheet_name='Optimized Schedule', index=False)
            build_machine_summary(schedule_df).to_excel(writer, sheet_name='Machine Summary', index=False)
            build_job_summary(schedule_df, data).to_excel(writer, sheet_name='Job Summary', index=False)
        metrics.to_excel(writer, sheet_name='Final Metrics', index=False)

    with pd.ExcelWriter(OUTPUT_VALIDATION_FILE, engine='openpyxl') as writer:
        for name, df in validation_dfs.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)


def export_gantt_html(schedule_df, output_html_path):
    if schedule_df.empty:
        with open(output_html_path, 'w', encoding='utf-8') as f:
            f.write('<html><body><h2>No feasible solution — no Gantt to display.</h2></body></html>')
        return

    palette = {'Op.10': '#3b82f6', 'Op.10 Split': '#f59e0b',
               'Op.20': '#10b981', 'Op.20 Split': '#f97316', 'Setup': '#111111'}
    records = []
    for _, task in schedule_df.iterrows():
        split_flag = int(task['split_count']) > 1
        leg = ('Op.10 Split' if task['operation'] == 10 and split_flag else
               'Op.10' if task['operation'] == 10 else
               'Op.20 Split' if split_flag else 'Op.20')
        label = f"{task['part_no']} S{int(task['split_id'])}/{int(task['split_count'])} | Op{int(task['operation'])}"
        if task['setup_min'] > EPS:
            records.append({'Legend': 'Setup', 'Machine': task['machine'],
                            'Start_min': task['setup_start_min'], 'Finish_min': task['setup_finish_min'],
                            'Duration_min': task['setup_min'], 'Text': '', 'Task': f'SETUP->{label}',
                            'Job': task['job_id'], 'Part': task['part_no'], 'Split': task['split_id'],
                            'Operation': task['operation'], 'Qty': task['quantity'],
                            'Setup_min': task['setup_min'], 'Processing_min': 0,
                            'Start_time': str(task['setup_start']), 'Finish_time': str(task['setup_finish'])})
        records.append({'Legend': leg, 'Machine': task['machine'],
                        'Start_min': task['start_min'], 'Finish_min': task['finish_min'],
                        'Duration_min': task['processing_min'], 'Text': label, 'Task': label,
                        'Job': task['job_id'], 'Part': task['part_no'], 'Split': task['split_id'],
                        'Operation': task['operation'], 'Qty': task['quantity'],
                        'Setup_min': task['setup_min'], 'Processing_min': task['processing_min'],
                        'Start_time': str(task['start']), 'Finish_time': str(task['finish'])})

    gantt_df    = pd.DataFrame(records).sort_values(['Machine', 'Start_min']).reset_index(drop=True)
    mach_order  = sorted(gantt_df['Machine'].unique())
    leg_order   = ['Op.10', 'Op.10 Split', 'Op.20', 'Op.20 Split', 'Setup']
    makespan    = schedule_df['finish_min'].max()

    fig = go.Figure()
    for leg in leg_order:
        sub = gantt_df[gantt_df['Legend'] == leg]
        if sub.empty: continue
        fig.add_trace(go.Bar(
            x=sub['Duration_min'].tolist(), y=sub['Machine'].tolist(),
            base=sub['Start_min'].tolist(), orientation='h', name=leg,
            marker=dict(color=palette[leg], line=dict(color='#111', width=1.4 if 'Split' in leg else 0.4),
                        pattern=dict(shape='/' if 'Split' in leg else '')),
            text=sub['Text'].tolist(), textposition='inside', insidetextanchor='middle',
            textfont=dict(color='white', size=10),
            customdata=list(zip(sub['Job'], sub['Part'], sub['Split'], sub['Operation'],
                                sub['Qty'], sub['Setup_min'], sub['Processing_min'],
                                sub['Start_min'], sub['Finish_min'], sub['Start_time'], sub['Finish_time'], sub['Task'])),
            hovertemplate=('Makine: %{y}<br>Task: %{customdata[11]}<br>Part: %{customdata[1]}<br>'
                           'Qty: %{customdata[4]}<br>Processing: %{customdata[6]:.1f} min<br>'
                           'Start: %{customdata[9]}<br>Finish: %{customdata[10]}<extra></extra>')))

    fig.add_shape(type='line', x0=makespan, x1=makespan, y0=0, y1=1, xref='x', yref='paper',
                  line=dict(width=2, dash='dash', color='#dc2626'))
    fig.update_layout(barmode='overlay',
                      title='Exact Gurobi Full Model — Gantt Chart',
                      xaxis_title='Zaman (dk)', yaxis_title='Makine',
                      yaxis=dict(categoryorder='array', categoryarray=mach_order, autorange='reversed'),
                      plot_bgcolor='white', paper_bgcolor='white', hovermode='closest',
                      width=2500, height=max(650, 70 * len(mach_order)),
                      margin=dict(l=80, r=40, t=110, b=60))
    fig.write_html(str(output_html_path), include_plotlyjs='cdn', full_html=True)


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == '__main__':
    tracemalloc.start()
    process   = psutil.Process(os.getpid())
    t_start   = time_module.time()

    print('\nLoading data...')
    GLOBAL_DATA = build_problem_data(
        shipment_file=SHIPMENT_FILE, sdst_file=SDST_FILE,
        machine_group_file=MACHINE_GROUP_FILE, use_shipped_quantity=True)

    print(f"Jobs          : {len(GLOBAL_DATA['jobs'])}")
    print(f"Machines      : {sorted(GLOBAL_DATA['machine_to_group'].keys())}")
    print(f"Planning start: {GLOBAL_DATA['planning_start']}")

    print('\nSolving...')
    solution = solve_exact_gurobi(GLOBAL_DATA)

    cpu_sec  = time_module.time() - t_start
    _, peak  = tracemalloc.get_traced_memory(); tracemalloc.stop()
    peak_mb  = max(peak / 1024 / 1024, process.memory_info().rss / 1024 / 1024)

    print(f'\n{"="*55}')
    print(f'CPU Time    : {cpu_sec:.2f} sec  ({cpu_sec/3600:.3f} h)')
    print(f'Peak Memory : {peak_mb:.2f} MB  ({peak_mb/1024:.3f} GB)')
    print(f'Big-M used  : {BIG_M}')
    print(f'Tardiness   : {TARDINESS_LIMIT_DAYS} days = {TARDINESS_LIMIT_MIN:.0f} min')
    print(f'{"="*55}')

    print('\nExtracting solution...')
    schedule_df, split_df = extract_solution_tables(GLOBAL_DATA, solution)

    print('Building validation...')
    validation_dfs = build_validation_tables(GLOBAL_DATA, solution, schedule_df, split_df)

    print('Exporting Excel...')
    export_excel_outputs(GLOBAL_DATA, solution, schedule_df, split_df, validation_dfs, cpu_sec, peak_mb)

    print('Exporting Gantt...')
    export_gantt_html(schedule_df, OUTPUT_GANTT_FILE)

    print('\nDone.')
    print('Schedule Excel   :', OUTPUT_SCHEDULE_FILE)
    print('Validation Excel :', OUTPUT_VALIDATION_FILE)
    print('Gantt HTML       :', OUTPUT_GANTT_FILE)
    print('Gurobi log       :', GUROBI_LOG_FILE)
    if solution['sol_count'] > 0:
        print('Objective        :', solution['model'].ObjVal)
        print('MIP Gap          :', f"{solution['model'].MIPGap*100:.2f}%")
        print('Total OT (min)   :', schedule_df['overtime_min'].sum())
    else:
        print('No feasible solution found — check gurobi log for details.')


Using files:
SHIPMENT_FILE      = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\492-güncel sevkiyat.xlsx
SDST_FILE          = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\SDST.xlsx
MACHINE_GROUP_FILE = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\machine_group_data.xlsx
GUROBI_LOG_FILE    = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\gurobi_exact_full.log

Loading data...
Duplicate order rows aggregated: 28 -> 27 unique jobs
Jobs          : 27
Machines      : ['T.3', 'T.3.1', 'T.3.10', 'T.3.11', 'T.3.12', 'T.3.2', 'T.3.3', 'T.3.4', 'T.3.6', 'T.3.7', 'T.3.8', 'T.3.9']
Planning start: 2026-04-29 07:00:00

Solving...

Schedule horizon : 14280 min  (238.0 h)
Big-M used       : 50000  (revised from 10**7 = 10000000; 200x tighter)
Tasks            : 108
Set parameter Username
Set parameter LicenseID to value 2797590
Academic license - for non-commercial use only - expires 2027-03-24
Set parameter LogFile to value "C:\Us